# 🤖 Generative AI Foundations
## Module 1 | Day 1 – Day 5

This notebook covers the practical requirements of the **Generative AI Foundations** module.

### Topics Covered
- Generative AI fundamentals
- Large Language Models (LLMs)
- Tokens and tokenisation
- Context windows
- Transformers and attention
- GenAI model types
- Hallucinations and limitations
- AI bias and Responsible AI
- Prompt engineering experiments
- Gemini API integration
- Simple GenAI application architecture
- Interactive chatbot experiment

> **Security:** The Gemini API key is loaded through Google Colab Secrets. Never hard-code or publicly share your API key.


## 1. Learning Objectives

By completing this notebook, you will be able to:

1. Explain what Generative AI and LLMs are.
2. Understand tokens, context windows and Transformers at a high level.
3. Experiment with different prompting techniques.
4. Observe how prompt structure affects responses.
5. Understand hallucinations, bias and Responsible AI.
6. Interact with a Gemini model through an API.
7. Build a simple text-based GenAI chatbot.


## 2. Generative AI Fundamentals

**Generative AI** refers to AI systems that can generate new content such as text, images, audio, video and code.

### AI → ML → DL → GenAI

- **Artificial Intelligence (AI):** Broad field of machines performing tasks associated with intelligence.
- **Machine Learning (ML):** Systems learn patterns from data.
- **Deep Learning (DL):** ML using multi-layer neural networks.
- **Generative AI (GenAI):** Models that generate new content based on learned patterns.

### Common applications

| Area | Example output |
|---|---|
| Text | Articles, summaries, conversations |
| Image | Generated artwork |
| Code | Programs and code suggestions |
| Audio | Speech/music |
| Video | Generated video |
| Multimodal | Text + image + audio understanding |


## 3. Large Language Models

A **Large Language Model (LLM)** is a model trained on large amounts of text data to understand and generate language.

A simplified flow is:

**Prompt → Tokenisation → Model processing → Next-token prediction → Output**

LLMs are commonly used for:
- Question answering
- Summarisation
- Translation
- Coding
- Content generation
- Conversational assistants


## 4. Tokens and Tokenisation

A model does not process a sentence exactly like a human reading individual words. Text is converted into smaller units called **tokens**.

Example:

```text
Input text
    ↓
Tokenisation
    ↓
Tokens
    ↓
LLM
    ↓
Output tokens
    ↓
Text
```

Tokenisation affects:
- Input limits
- Output limits
- Context usage
- API cost in many systems
- Model processing


## 5. Context Windows

A **context window** is the amount of information a model can consider during a request or conversation.

Context can include:
- Current prompt
- Earlier conversation
- Documents
- Retrieved information
- Instructions

A larger context window can be useful for long documents and extended conversations, but it does not guarantee that every piece of information will be used perfectly.


## 6. Transformers and Attention

Modern LLMs are largely based on the **Transformer** architecture.

High-level flow:

```text
Text
 ↓
Tokens
 ↓
Embeddings
 ↓
Transformer layers
 ↓
Attention mechanisms
 ↓
Predictions
 ↓
Generated text
```

### Attention

Attention allows the model to consider relationships between different tokens when processing a sequence.

For example, in:

> "Rahul deposited money at the bank."

The surrounding words help the model interpret **bank** as a financial institution.

At this stage, focus on the concept rather than the mathematical implementation.


## 7. Types of Generative AI Models

| Model type | Typical output |
|---|---|
| Large Language Model | Text and code |
| Image generation model | Images |
| Audio generation model | Speech/music |
| Video generation model | Video |
| Multimodal model | Multiple modalities |

Examples of widely known model families include Gemini, GPT, Claude, Llama and Mistral. Model capabilities and available versions change over time.


## 8. Hallucinations

An AI **hallucination** is an output that may sound confident and plausible but is incorrect, unsupported or fabricated.

### Why this matters

AI-generated information should be checked when accuracy is important, especially for:
- Academic work
- Legal information
- Financial decisions
- Medical information
- Current events
- Technical facts

A good practice is to ask the model to distinguish uncertainty from verified information, then independently verify important claims.


## 9. AI Bias and Responsible AI

### Possible sources of bias
- Bias in training data
- Under-representation
- Historical bias
- Prompt/context effects

### Responsible AI principles
- Fairness
- Privacy
- Security
- Transparency
- Accountability
- Human oversight

Responsible AI means considering both the usefulness and potential risks of AI systems.


# 10. Setup: Gemini API

We will use Google's Python GenAI SDK available in the Google Colab environment.

The notebook uses the Gemini model:

`gemini-3.6-flash`

No package downgrade is required.

In [ ]:
import google.genai

print(
    "google-genai SDK:",
    getattr(google.genai, "__version__", "installed")
)

In [ ]:
from google import genai
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found. "
        "Add it to Google Colab Secrets and run this cell again."
    )

client = genai.Client(api_key=API_KEY)

print("Gemini API configured successfully.")

### 🔐 How to add the API key safely

In Google Colab:

1. Open the **Secrets** panel.
2. Add a new secret named `GEMINI_API_KEY`.
3. Paste your Gemini API key as the secret value.
4. Enable notebook access for the secret.
5. Run the setup cell above.

**Do not write the real API key directly inside a code cell.**


# 11. Experiment 1: Basic Prompt

This is a simple prompt without additional instructions.


## ⚠️ Error Fix Notes

### 1. Dependency Conflict

The earlier notebook used a fixed older version of `google-genai`, which conflicted with packages already installed in Google Colab.

This notebook does not force an older `google-genai` or `google-auth` version. It uses the SDK already available in the Colab environment.

### 2. 503 UNAVAILABLE

A `503 UNAVAILABLE` error means that the Gemini server is temporarily unavailable or experiencing high demand.

To handle this, the notebook automatically retries the request using the same model:

`gemini-3.6-flash`

If the service is temporarily unavailable, wait a few moments and run the cell again.

In [ ]:
MODEL_NAME = "gemini-3.6-flash"

print("Gemini model:", MODEL_NAME)

In [ ]:
import time
from google.genai import errors

def generate_with_retry(prompt, max_retries=3):
    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            print(f"Success with model: {MODEL_NAME}")
            return response

        except errors.ServerError as error:
            last_error = error
            message = str(error)

            if (
                "503" in message
                or "UNAVAILABLE" in message
                or "high demand" in message
            ):
                wait_seconds = 2 ** attempt

                print(
                    f"Gemini 3.6 Flash is temporarily unavailable."
                )
                print(
                    f"Retrying in {wait_seconds} seconds..."
                )

                time.sleep(wait_seconds)
                continue

            raise

    raise RuntimeError(
        f"Gemini 3.6 Flash is currently unavailable. "
        f"Last error: {last_error}"
    )

# 12. Experiment 2: Role Prompting

Here we give the model a role and specify the target audience.


In [ ]:
prompt = """
You are a computer science teacher.

Explain Generative AI to a beginner.
Use simple language and two real-world examples.
"""

response = generate_with_retry(prompt)

print(response.text)


# 13. Experiment 3: Structured Prompting

A structured prompt specifies exactly how the answer should be organised.


In [ ]:
prompt = """
Explain Large Language Models using exactly these sections:

1. Definition
2. How they work
3. Applications
4. Advantages
5. Limitations

Keep the explanation beginner-friendly.
"""

response = generate_with_retry(prompt)

print(response.text)

# 14. Experiment 4: Few-Shot Prompting

Few-shot prompting provides examples so the model can infer the expected format.


In [ ]:
prompt = """
Convert the abbreviations into their full forms.

Examples:
AI → Artificial Intelligence
ML → Machine Learning

Now complete:

LLM →
NLP →
API →
RAG →
"""

response = generate_with_retry(prompt)

print(response.text)

# 15. Experiment 5: Summarisation

The model can transform a longer piece of text into a concise summary.


In [ ]:
text = """
Generative AI is a branch of artificial intelligence that can create new
content such as text, images, audio, video and code. Large Language Models
are a major type of Generative AI system that can understand and generate
human-like text. These systems can be useful for writing, coding,
summarisation and conversational applications, but their outputs can
sometimes contain inaccurate information.
"""

prompt = f"""
Summarise the following text in exactly 3 simple bullet points:

{text}
"""

response = generate_with_retry(prompt)

print(response.text)

# 16. Experiment 6: Hallucination Awareness

This experiment demonstrates why generated answers should not automatically be treated as verified facts.


In [ ]:
prompt = """
A fictional programming language called XYZPython was supposedly invented
in 1895.

Explain whether this claim can be verified from the information given.

Do not invent a historical source or person.

Clearly distinguish the fictional premise from verified information.
"""

response = generate_with_retry(prompt)

print(response.text)

In [ ]:
print("Observation:")
print("A responsible AI workflow should separate generated claims from verified facts.")
print("Important claims should be checked against reliable sources.")


# 17. Experiment 7: Responsible AI

Ask the model to explain the main principles of responsible AI.


In [ ]:
prompt = """
Explain Responsible AI for a beginner.

Cover:

1. Fairness
2. Privacy
3. Security
4. Transparency
5. Accountability
6. Human oversight

Give one practical example for each.
"""

response = generate_with_retry(prompt)

print(response.text)

# 18. Experiment 8: GenAI Application Architecture

A basic API-based GenAI application can be represented as:

```text
User
  ↓
User Interface
  ↓
Prompt Handling
  ↓
GenAI Application
  ↓
Gemini API
  ↓
Gemini 3.6 Flash
  ↓
Generated Response
  ↓
Response Processing
  ↓
User
```

The Streamlit application in the companion project follows this general architecture.


# 19. Experiment 9: Reusable AI Function

Create a simple function that sends a question to Gemini.


In [ ]:
def ask_ai(question):
    response = generate_with_retry(question)
    return response.text

answer = ask_ai(
    "What is a Transformer in Artificial Intelligence?"
)

print(answer)

# 20. Experiment 10: Interactive Chatbot

Run the following cell and type questions in the input box. Type `exit` to stop.

This is a simple demonstration. It sends each question independently rather than maintaining a full conversation history.


In [ ]:
print("🤖 Gemini 3.6 Flash Chatbot")
print("Type 'exit' to stop.\n")

while True:
    question = input("You: ")

    if question.lower().strip() == "exit":
        print("Chat ended.")
        break

    try:
        answer = ask_ai(question)

        print("\nAI:", answer)
        print()

    except Exception as error:
        print("\nError:", error)
        print("Please wait a moment and try again.\n")

# 21. AI Model Comparison Experiment

Use the same prompt in multiple AI systems available to you and record the results.

### Prompt

```text
Explain Transformers to a first-year computer science student
using a simple real-world analogy.
```

### Record observations

| Model | Clarity | Detail | Structure | Accuracy observations |
|---|---|---|---|---|
| Gemini | | | | |
| ChatGPT | | | | |
| Claude / another model | | | | |

The Gemini API experiments in this notebook use only Gemini 3.6 Flash.

The purpose of this comparison is to observe differences in generated
responses rather than declaring one model universally superior.


# 22. Key Learnings

Complete these statements after running the experiments:

- Generative AI is: ______________________________
- An LLM is: ____________________________________
- A token is: ____________________________________
- A context window is: ___________________________
- Transformers are important because: ____________
- Hallucinations mean: ___________________________
- Responsible AI requires: _______________________
- Prompt engineering helps by: ___________________
- An API allows an application to: _______________


# 23. Final Conclusion

This module introduced the foundations of Generative AI and Large Language
Models and demonstrated how to interact with **Gemini 3.6 Flash** through
the Gemini API.

### Practical Work Completed

- ✅ Generative AI fundamentals
- ✅ LLM fundamentals
- ✅ Tokens and context windows
- ✅ Transformers and attention
- ✅ GenAI model types
- ✅ Hallucination awareness
- ✅ AI bias and Responsible AI
- ✅ Prompt engineering
- ✅ Gemini 3.6 Flash API integration
- ✅ Basic prompt experiment
- ✅ Role prompting
- ✅ Structured prompting
- ✅ Few-shot prompting
- ✅ Summarisation
- ✅ Reusable AI function
- ✅ Interactive chatbot

## Module 1: Generative AI Foundations — Completed 🚀
